In [9]:
import glob
import pandas as pd

In [10]:
RESULT_DIR = 'aggregated/'
DATA_DIR = 'sentimental_flair/'
NEG = -1
LEFT = 0
RIGHT = 1
LOW = 0
HIGH = 1

In [11]:
# https://www.geeksforgeeks.org/getting-all-csv-files-from-a-directory-using-python/
# csv files in the path 
files = glob.glob(DATA_DIR + "/*.csv") 
  
# defining an empty list to store  
# content 
df = pd.DataFrame() 
content = [] 
  
# checking all the csv files in the  
# specified path 
for filename in files: 
    
    # reading content of csv file 
    # content.append(filename) 
    df = pd.read_csv(filename, index_col=None) 
    content.append(df) 
  
# converting content to data frame 
df = pd.concat(content) 
df.reset_index(inplace=True)
# df.to_csv(DATA_DIR+"combined_headlines.csv", index=False)
print(df.columns) 

Index(['level_0', 'Headline', 'Source', 'Year', 'Month', 'Day', 'Date_id',
       'Pre-Covid', 'Bias', 'Cred', 'Sentiment', 'index'],
      dtype='object')


In [28]:
result_df = pd.DataFrame(columns=['date','negative_l','total_l','negative_r','total_r'])
result_df = pd.DataFrame({
                        'date': pd.Series(dtype='str'),
                        'pre_covid': pd.Series(dtype='int'),
                        'negative_l': pd.Series(dtype='int'),
                        'total_l': pd.Series(dtype='int'),
                        'negative_r': pd.Series(dtype='int'),
                        'total_r': pd.Series(dtype='int'),
                        'negative_low': pd.Series(dtype='int'),
                        'total_low': pd.Series(dtype='int'),
                        'negative_high': pd.Series(dtype='int'),
                        'total_high': pd.Series(dtype='int'),
                        'total': pd.Series(dtype='int'),
                        })
result_df

,date,pre_covid,negative_l,total_l,negative_r,total_r,negative_low,total_low,negative_high,total_high,total


In [22]:
def addInitialRowBiasCred(bias, cred, sentiment):
  row_data = []
  isNeg = 1 if sentiment == -1 else 0
  if bias == LEFT:
      row_data.extend([isNeg, 1, 0, 0])
  elif bias == RIGHT:
      row_data.extend([0, 0, isNeg, 1])

  if cred == LOW:
    row_data.extend([isNeg, 1, 0, 0])
  elif cred == HIGH:
    row_data.extend([0, 0, isNeg, 1])
  
  return row_data
     

In [29]:
df = pd.read_csv('combined_headlines.csv')
for ind in df.index:
    date = str(df['Year'][ind]) + '-' + str(df['Month'][ind]) + '-' + str(df['Day'][ind])
    pre_covid = df['Pre-Covid'][ind]
    bias = df['Bias'][ind]
    sentiment = df['Sentiment'][ind]
    credibility = df['Cred'][ind]

    selected_df = result_df[result_df['date'] == date]
    if selected_df.empty: # new date for result_df
        row_data = []
        row_data.append(date) # date_id
        row_data.append(pre_covid) # pre_covid
        row_data.extend(addInitialRowBiasCred(bias, credibility, sentiment))
        row_data.append(1)  # to total
        row = pd.Series(row_data, index=result_df.columns)
        result_df = pd.concat([result_df, pd.DataFrame([row])], ignore_index=True)
    else: # date exists
        row_idx = result_df.index[result_df['date'] == date].tolist()[0]
        if bias == LEFT:
            if sentiment == NEG:
                result_df.loc[row_idx, ['negative_l']] = result_df.loc[row_idx, ['negative_l']] + 1
                result_df.loc[row_idx, ['total_l']] = result_df.loc[row_idx, ['total_l']] + 1
            else:
                result_df.loc[row_idx, ['total_l']] = result_df.loc[row_idx, ['total_l']] + 1
        elif bias == RIGHT:
            if sentiment == NEG:
                result_df.loc[row_idx, ['negative_r']] = result_df.loc[row_idx, ['negative_r']] + 1
                result_df.loc[row_idx, ['total_r']] = result_df.loc[row_idx, ['total_r']] + 1
            else:
                result_df.loc[row_idx, ['total_r']] = result_df.loc[row_idx, ['total_r']] + 1
        
        if credibility == LOW:
            if sentiment == NEG:
                result_df.loc[row_idx, ['negative_low']] = result_df.loc[row_idx, ['negative_low']] + 1
                result_df.loc[row_idx, ['total_low']] = result_df.loc[row_idx, ['total_low']] + 1
            else:
                result_df.loc[row_idx, ['total_low']] = result_df.loc[row_idx, ['total_low']] + 1
        elif credibility == HIGH:
            if sentiment == NEG:
                result_df.loc[row_idx, ['negative_high']] = result_df.loc[row_idx, ['negative_high']] + 1
                result_df.loc[row_idx, ['total_high']] = result_df.loc[row_idx, ['total_high']] + 1
            else:
                result_df.loc[row_idx, ['total_high']] = result_df.loc[row_idx, ['total_high']] + 1

        result_df.loc[row_idx, ['total']] = result_df.loc[row_idx, ['total']] + 1 

    
    

In [30]:
date_id = result_df.index
result_df.insert(0, 'date_id', date_id)
result_df

,date_id,date,pre_covid,negative_l,total_l,negative_r,total_r,negative_low,total_low,negative_high,total_high,total
0,0,2018-1-2,0,22,43,39,74,14,24,47,93,117
1,1,2018-1-9,0,24,41,45,88,22,38,47,91,129
2,2,2018-2-8,0,30,46,51,93,27,43,54,96,139
3,3,2018-3-14,0,26,40,58,98,25,48,59,90,138
4,4,2018-4-10,0,29,42,53,100,29,51,53,91,142
...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,2021-9-18,1,26,44,14,29,0,0,40,73,73
96,96,2021-10-10,1,19,56,26,50,0,0,45,106,106
97,97,2021-11-5,1,25,48,50,96,26,46,49,98,144
98,98,2021-11-16,1,27,47,49,93,26,43,50,97,140


In [32]:
result_file = RESULT_DIR + 'aggregated_full.csv'
result_df.to_csv(result_file, index=False)